# Model Comparison with SMOTE

This notebook trains baseline models and compares performance after applying SMOTE to handle class imbalance.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
import joblib

In [ ]:
BASE_FEATURES = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
raw_path = Path("data/creditcard.csv")
proc_path = Path("data/processed.csv")

if proc_path.exists():
    df = pd.read_csv(proc_path)
elif raw_path.exists():
    df = pd.read_csv(raw_path)
else:
    raise FileNotFoundError("Missing data file. Place creditcard.csv in data/ or run notebooks/02_features.py.")

required = BASE_FEATURES + ["Class"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required]
X = df[BASE_FEATURES]
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_smote).value_counts().to_dict())

In [ ]:
def evaluate_model(name, model, X_test_scaled, y_test):
    proba = model.predict_proba(X_test_scaled)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        "model": model,
        "roc_auc": roc_auc_score(y_test, proba),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

models = {
    "logistic": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    results[name] = evaluate_model(name, model, X_test_scaled, y_test)

for name, res in results.items():
    print(
        f"{name}: ROC-AUC={res['roc_auc']:.4f}, "
        f"Precision={res['precision']:.4f}, Recall={res['recall']:.4f}, F1={res['f1']:.4f}"
    )

best_name = max(results, key=lambda k: results[k]["roc_auc"])
best_model = results[best_name]["model"]
print(f"Best model: {best_name}")

In [ ]:
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, models_dir / "fraud_model.pkl")
joblib.dump(scaler, models_dir / "scaler.pkl")
print("Saved model to models/fraud_model.pkl")
print("Saved scaler to models/scaler.pkl")